# 03 - Benchmark Hunyuan3D-2mini trên T4 (29 vật thể GSO)

**Mô hình**: Hunyuan3D-2mini (shape generator 0.6B) — chỉ tạo **hình dạng**, không tạo texture (đúng yêu cầu nghiên cứu).
**Repo**: `Tencent-Hunyuan/Hunyuan3D-2` @ commit `f8db63096c8282cb27354314d896feba5ba6ff8a`.
**Trọng số** (tải tự động từ Hugging Face): `tencent/Hunyuan3D-2mini`, subfolder `hunyuan3d-dit-v2-mini`.

**Giao thức**: seed 42; `num_inference_steps=30`, `octree_resolution=380`, `num_chunks=20000` (mặc định chính thức của 2mini);
input = adapter `hunyuan3d2mini` (RGBA 512×512); output `output_type='trimesh'` → xuất `.glb` chuẩn.

**Cấu trúc**: PHẦN 1–2 kiểm tra GPU/Drive; PHẦN 3 cài đặt; PHẦN 4 nạp mô hình + smoke test → **DỪNG chờ duyệt**; PHẦN 5 batch 29 ảnh.

## PHẦN 1 - Kiểm tra GPU

In [ ]:
import subprocess, torch, platform, time

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print('python:', platform.python_version())
print('torch:', torch.__version__, '| cuda:', torch.version.cuda, '| available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('KHONG CO CUDA - bam Runtime > Disconnect, chon GPU T4 roi chay lai.')
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name, '| T4?', 'T4' in gpu_name)
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## PHẦN 2 - Mount Drive + cấu hình

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, csv, json, hashlib, zipfile
import numpy as np

DRIVE_ROOT = '/content/drive/MyDrive/nien_luan_3d_benchmark'
DATASET_DIR = os.path.join(DRIVE_ROOT, 'dataset')
RUNS_DIR = os.path.join(DRIVE_ROOT, 'runs')

MODEL_ID = 'hunyuan3d2mini'
REPO_URL = 'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git'
REPO_COMMIT = 'f8db63096c8282cb27354314d896feba5ba6ff8a'
HF_WEIGHTS = 'tencent/Hunyuan3D-2mini'
HF_SUBFOLDER = 'hunyuan3d-dit-v2-mini'
SMOKE_ID = 'Hefty_Waste_Basket_Decorative_Bronze_85_liter'
SEED = 42
NUM_INFERENCE_STEPS = 30
OCTREE_RESOLUTION = 380
NUM_CHUNKS = 20000

RUN_ID = '%s_t4_%s' % (MODEL_ID, time.strftime('%Y%m%d_%H%M%S'))
RUN_DIR = os.path.join(RUNS_DIR, RUN_ID)
MESH_DIR = os.path.join(RUN_DIR, 'meshes')
os.makedirs(MESH_DIR, exist_ok=True)
print('RUN_ID:', RUN_ID)

In [ ]:
manifest = {}
with open(os.path.join(DATASET_DIR, 'dataset_manifest.csv'), encoding='utf-8-sig') as f:
    for r in csv.DictReader(f):
        manifest[r['object_id']] = r
expected_ids = sorted(manifest.keys())
print('Manifest:', len(expected_ids), 'objects')
assert len(expected_ids) == 29 and SMOKE_ID in expected_ids

adapter_dir = os.path.join(DATASET_DIR, 'adapters', 'hunyuan3d2mini')
missing = [oid for oid in expected_ids if not os.path.exists(os.path.join(adapter_dir, oid + '.png'))]
assert not missing, 'Thieu adapter hunyuan3d2mini - chay notebook 00 truoc!'
print('Adapter hunyuan3d2mini: ok')

## PHẦN 3 - Cài đặt repo + dependencies

Clone `Tencent-Hunyuan/Hunyuan3D-2` @ commit cố định, cài package `hy3dgen` (chỉ shape — **không** cài phần texture rasterizer).
Không dùng `requirements.txt` của repo (tránh pin phiên bản không tương thích Python 3.12 của Colab) — `pip install -e .` tự kéo đủ dependencies chính thức.
Lần đầu sẽ tải trọng số ~ vài GB từ Hugging Face — chờ cho xong.

In [ ]:
if not os.path.exists('/content/Hunyuan3D-2/.git'):
    !git clone --quiet {REPO_URL} /content/Hunyuan3D-2
%cd /content/Hunyuan3D-2
!git checkout --quiet {REPO_COMMIT}
!git log -1 --format='%H %s'
!git rev-parse HEAD

In [ ]:
!pip install -q -e .
!pip install -q "scipy==1.14.1"
print('deps ok')

In [ ]:
%cd /content/Hunyuan3D-2
import torch
from PIL import Image

t0 = time.perf_counter()
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    HF_WEIGHTS, subfolder=HF_SUBFOLDER, use_safetensors=True, device='cuda'
)
torch.cuda.empty_cache()
torch.cuda.synchronize()
model_load_time_s = round(time.perf_counter() - t0, 3)
print('pipeline ready. model load time_s:', model_load_time_s)

In [ ]:
import trimesh

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest().upper()

def run_one(oid, mesh_dir):
    row = {'run_id': RUN_ID, 'model': MODEL_ID, 'object_id': oid,
           'commit': REPO_COMMIT, 'weights': HF_WEIGHTS, 'subfolder': HF_SUBFOLDER,
           'adapter': 'hunyuan3d2mini_rgba', 'num_inference_steps': NUM_INFERENCE_STEPS,
           'octree_resolution': OCTREE_RESOLUTION, 'num_chunks': NUM_CHUNKS, 'seed': SEED,
           'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'), 'status': 'failed',
           'success': False, 'error_message': '', 'n_vertices': '', 'n_triangles': '',
           'mesh_bytes': '', 'dominant_axis': ''}
    try:
        t_pre = time.perf_counter()
        input_image = Image.open(os.path.join(adapter_dir, oid + '.png')).convert('RGBA')
        t_pre_el = round(time.perf_counter() - t_pre, 3)

        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

        t_inf = time.perf_counter()
        mesh_trimesh = pipeline(
            image=input_image, num_inference_steps=NUM_INFERENCE_STEPS,
            octree_resolution=OCTREE_RESOLUTION, num_chunks=NUM_CHUNKS,
            generator=torch.manual_seed(SEED), output_type='trimesh',
        )[0]
        torch.cuda.synchronize()
        t_inf_el = round(time.perf_counter() - t_inf, 3)
        vram_peak_mb = round(torch.cuda.max_memory_allocated() / 1e6, 1)

        t_post = time.perf_counter()
        glb_path = os.path.join(mesh_dir, oid + '.glb')
        mesh_trimesh.export(glb_path)
        t_post_el = round(time.perf_counter() - t_post, 3)

        ext = mesh_trimesh.bounds[1] - mesh_trimesh.bounds[0]
        dominant = ['x', 'y', 'z'][int(np.argmax(ext))]

        row.update({'preprocessing_time_s': t_pre_el, 'inference_time_s': t_inf_el,
                    'postprocessing_time_s': t_post_el,
                    'total_time_s': round(t_pre_el + t_inf_el + t_post_el, 3),
                    'vram_peak_mb': vram_peak_mb,
                    'n_vertices': int(mesh_trimesh.vertices.shape[0]),
                    'n_triangles': int(mesh_trimesh.faces.shape[0]),
                    'mesh_bytes': os.path.getsize(glb_path), 'dominant_axis': dominant,
                    'status': 'success', 'success': True})
    except Exception as e:
        row['error_message'] = '%s: %s' % (type(e).__name__, e)
    return row

print('Ham da san sang.')

## PHẦN 4 - SMOKE TEST

Chạy 1 ảnh (`Hefty_Waste_Basket_Decorative_Bronze_85_liter`). Sau cell này **DỪNG LẠI, gửi kết quả về để duyệt**.

In [ ]:
print('SMOKE TEST object:', SMOKE_ID)
row = run_one(SMOKE_ID, MESH_DIR)
for k, v in row.items():
    print('%-24s %s' % (k, v))

if row['success']:
    if row['dominant_axis'] != 'y':
        print('LUU Y: truc chinh cua mesh la %s (khong phai y) - bao lai khi gui ket qua.' % row['dominant_axis'])
    print('SMOKE: SUCCESS')
else:
    print('SMOKE: FAIL ->', row['error_message'])
    raise SystemExit('SMOKE FAIL - DUNG LAI, gui man hinh ve.')

## ⛔ DỪNG LẠI — chờ duyệt

1. Chụp màn hình kết quả (GPU, thời gian inference, VRAM, dominant_axis, SMOKE: SUCCESS).
2. Gửi về. **KHÔNG chạy PHẦN 5 trước khi duyệt.**

## PHẦN 5 - BATCH 29 ẢNH (chỉ sau khi duyệt)

In [ ]:
log_csv = os.path.join(RUN_DIR, 'run_log.csv')
fields = ['run_id', 'model', 'object_id', 'commit', 'weights', 'subfolder', 'adapter',
          'num_inference_steps', 'octree_resolution', 'num_chunks', 'seed', 'timestamp',
          'preprocessing_time_s', 'inference_time_s', 'postprocessing_time_s',
          'total_time_s', 'vram_peak_mb', 'n_vertices', 'n_triangles', 'mesh_bytes',
          'dominant_axis', 'status', 'success', 'error_message']

done = set()
if os.path.exists(log_csv):
    with open(log_csv, encoding='utf-8-sig') as f:
        for r in csv.DictReader(f):
            if r.get('success') == 'True':
                done.add(r['object_id'])

def append_row(row):
    new = not os.path.exists(log_csv)
    with open(log_csv, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        if new:
            w.writeheader()
        w.writerow(row)

t_batch = time.perf_counter()
for i, oid in enumerate(sorted(manifest.keys())):
    if oid in done:
        print('[%02d/%02d] SKIP (da xong): %s' % (i + 1, 29, oid))
        continue
    row = run_one(oid, MESH_DIR)
    append_row(row)
    tag = 'OK ' if row['success'] else 'ERR'
    print('[%02d/%02d] %s %s  inf=%ss  total=%ss vram=%sMB tris=%s' % (
        i + 1, 29, tag, oid, row.get('inference_time_s'), row.get('total_time_s'),
        row.get('vram_peak_mb'), row.get('n_triangles')))
    if not row['success']:
        print('      ->', row['error_message'])

batch_el = round(time.perf_counter() - t_batch, 3)
print('Batch time_s:', batch_el)
print('Done. Log:', log_csv)

In [ ]:
ok = 0
fail = 0
with open(log_csv, encoding='utf-8-sig') as f:
    for r in csv.DictReader(f):
        if r['success'] == 'True':
            ok += 1
        else:
            fail += 1

meta = {
    'run_id': RUN_ID, 'model': MODEL_ID, 'repo': REPO_URL, 'commit': REPO_COMMIT,
    'weights': HF_WEIGHTS, 'subfolder': HF_SUBFOLDER,
    'seed': SEED, 'num_inference_steps': NUM_INFERENCE_STEPS,
    'octree_resolution': OCTREE_RESOLUTION, 'num_chunks': NUM_CHUNKS,
    'adapter': 'hunyuan3d2mini_rgba', 'input_size': 512, 'shape_only': True,
    'gpu': gpu_name, 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'python': platform.python_version(),
    'time_start': t0, 'time_end': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'model_load_time_s': model_load_time_s,
    'objects_total': 29, 'objects_success': ok, 'objects_failed': fail,
    'batch_time_s': batch_el,
    'note': 'single run fixed seed; shape only (no texture); no absolute comparison claims'
}
with open(os.path.join(RUN_DIR, 'metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

zip_p = os.path.join(RUN_DIR, RUN_ID + '.zip')
with zipfile.ZipFile(zip_p, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in ['run_log.csv', 'metadata.json']:
        zf.write(os.path.join(RUN_DIR, fn), fn)
    for fn in sorted(os.listdir(MESH_DIR)):
        zf.write(os.path.join(MESH_DIR, fn), 'meshes/' + fn)

print('success=%d fail=%d' % (ok, fail))
print('metadata:', os.path.join(RUN_DIR, 'metadata.json'))
print('zip:', zip_p, os.path.getsize(zip_p), 'bytes')
print('CHAY XONG. Gui man hinh ket qua + ten run_id ve.')